In [1]:
##this is the second notebook which makes adds mnpass to the base scenario 
# BASE missing road, corrections, attribute cards are run in previous notebook CreateVer01 
## optional base corrections temp folder for testing project cards 
## BaseCorrectionsTemp cards should be moved into base corrections once they are tested 

In [2]:
import os
import sys
import pickle
import pandas as pd

import numpy as np

from pyproj import CRS

from network_wrangler import load_roadway
from network_wrangler import load_transit
from network_wrangler import create_scenario
from network_wrangler import Scenario
from network_wrangler.roadway import write_roadway
from network_wrangler.transit import write_transit

from met_council_wrangler import MetCouncil_Parameters
from met_council_wrangler import metcouncil_roadway
from met_council_wrangler import metcouncil_transit

from cube_wrangler import Parameters
from cube_wrangler import util
from cube_wrangler import roadway
from cube_wrangler import StandardTransit

Geopandas is not using pyogrio as the I/O engine.                Install pyogrio to benefit from faster I/O.


In [3]:
import logging
logger = logging.getLogger("WranglerLogger")

if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    logger.addHandler(handler)

logger.handlers[0].stream = sys.stdout
logger.setLevel(logging.INFO)

In [4]:
%reload_ext autoreload
%autoreload 2

# remote i/o

In [5]:
metcouncil_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler")
cube_wrangler_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler")

cc_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\standard_networks\v00\standard_networks")

net_dir = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks")
output_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output")

In [6]:
### base corrections temp only if testing cards that included in notebook one 
# project_card_dir = os.path.join("C:/project_card_registry/projects/BaseCorrectionsTemp")
## base mnpass
project_card_dir1 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1")
## no build cards
project_card_dir2 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\NoBuild_TPP2023_v1")
project_card_dir3 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\NoBuild_MnPASS2023_v1")
## build scenario 
project_card_dir4 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\Build_TPP2050_v1")
project_card_dir5 = os.path.join(r"Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\Build_MnPASS2050_v1")
####temp space if needed
# project_card_dir6= os.path.join("C:/project_card_registry/projects/test")

In [7]:
metcouncil_parameters = MetCouncil_Parameters(
    metcouncil_wrangler_base_dir=metcouncil_wrangler_dir,
    cube_wrangler_base_dir=cube_wrangler_dir
)

cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


# Load Version01

version 01 has the standard networks with base corrections, rail links and nodes, external stations and connectors

In [8]:
link_file = os.path.join(net_dir, 'v01', 'standard_networks', 'link.json')
node_file = os.path.join(net_dir, 'v01', 'standard_networks', 'node.geojson')
shape_file = os.path.join(net_dir, 'v01', 'standard_networks', 'shape.geojson')

roadway_net = load_roadway(
    links_file=link_file,
    nodes_file=node_file,
    shapes_file=shape_file,
)

transit_net = load_transit(os.path.join(net_dir, 'v01', 'standard_networks'))

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Read 414282 nodes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\node.geojson in 24.33.
Reading links from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\link.json.
Read + transformed 1061756 links from             Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\link.json in 135.47.
Reading GTFS feed tables from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks
Initializing frequencies


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\transit\io.py:86: DtypeWarning: Columns (1) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(file)


PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Initializing routes
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Initializing shapes
Initializing stops
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Initializing trips
Referencing table stop_times not yet set in        

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\engines\pandas_engine.py:873: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  col = to_datetime_fn(col, **self.to_datetime_kwargs)


In [9]:
base_scenario = {"road_net": roadway_net, "transit_net": transit_net}
version_01_scenario = create_scenario(base_scenario = base_scenario)

Creating Scenario
Base_scenario doesn't contain ['road_net', 'transit_net', 'applied_projects', 'conflicts']
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table st

# Create Scenario 02

In [10]:
version_02_scenario = create_scenario(
    base_scenario = version_01_scenario,
    project_card_filepath = project_card_dir1
)

Creating Scenario
PK table trips for specified FK                     frequencies.trip_id not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
PK table agencies for specified FK                     routes.agency_id not in table list - skipping validation.
Referencing table trips not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
PK table stops for specified FK                     stops.parent_station not in <class 'network_wrangler.transit.feed.feed.Feed'>-                      skipping validation.
Referencing table stops not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Feed'> - skipping fk validation.
Referencing table stop_times not yet set in                      <class 'network_wrangler.transit.feed.feed.Fee

In [11]:
# applying basemnpass and corrections cards before future cards to avoid mnpass dependency issues -
version_02_scenario.apply_all_projects()

Applying add mnpass_code from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1\MnPASSCODE.yml
Final selected links: 1061756
Applying 394 wb managed lane from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1\394_WB_MnPASS.yml
Selecting using explicit link identifiers.
Final selected links: 14


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Access point not set in project card for a new managed lane.                                   
Setting ML_access_point to True for selected links.
Egress point not set in project card for a new managed lane.                                   
Setting ML_egress_point to True for selected links.
Applying 394 reverse lane onramp mnpasspay from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1\394_Reverse_lane_onramp_MnPASS.yml


\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\roadway\links\edit.py:329: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'all' has dtype incompatible with bool, please explicitly cast to a compatible dtype first.
  links_df.loc[link_idx, prop_name] = prop_change.set


Selecting using explicit link identifiers.
Final selected links: 3
Selecting using explicit link identifiers.
Final selected links: 2
Applying 394 reverse lanes by time period from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1\394_Reverse_lanes_by_time_period.yml
Selecting using explicit link identifiers.
Final selected links: 1
Selecting using explicit link identifiers.
Final selected links: 3
Selecting using explicit link identifiers.
Final selected links: 1
Selecting using explicit link identifiers.
Final selected links: 3
Selecting using explicit link identifiers.
Final selected links: 1
Selecting using explicit link identifiers.
Final selected links: 4
Selecting using explicit link identifiers.
Final selected links: 5
Applying 394 eb managed lane from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPAS

...

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Selecting using explicit link identifiers.
Final selected links: 22
Access point not set in project card for a new managed lane.                                   
Setting ML_access_point to True for selected links.
Egress point not set in project card for a new managed lane.                                   
Setting ML_egress_point to True for selected links.
Applying 35e nb managed lane from file:                            Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\project_card\BaseMnPASS_v1\35E_NB_MnPASS.yml


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Selecting using explicit link identifiers.
Final selected links: 23
Access point not set in project card for a new managed lane.                                   
Setting ML_access_point to True for selected links.
Egress point not set in project card for a new managed lane.                                   
Setting ML_egress_point to True for selected links.


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


In [12]:
version_02_scenario.applied_projects

['add mnpass_code',
 '394 wb managed lane',
 '394 reverse lane onramp mnpasspay',
 '394 reverse lanes by time period',
 '394 eb managed lane',
 '35w south sb managed lane',
 '35w south nb managed lane',
 '35w north sb managed lane',
 '35w north nb managed lane',
 '35e sb managed lane',
 '35e nb managed lane']

# Save version 02 standard networks

In [13]:
write_roadway(
    version_02_scenario.road_net, 
    file_format="geojson", 
    out_dir= os.path.join(net_dir, 'v02base', 'standard_networks'), 
    overwrite=True, 
    convert_complex_link_properties_to_single_field=True
)
write_transit(version_02_scenario.transit_net, file_format="txt", out_dir= os.path.join(net_dir, 'v02base', 'standard_networks'), overwrite=True)

Read 556559 shapes from Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v01\standard_networks\shape.geojson in 28.64.
Wrote 6 files to Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\Networks\standard_networks\v02base\standard_networks


# Make Travel Model Network

### Add centroid and centroid connectors

In [14]:
r_net = metcouncil_roadway.add_centroid_and_centroid_connector(
    roadway_network = version_02_scenario.road_net,
    parameters = metcouncil_parameters,
    centroid_file = os.path.join(cc_dir, 'centroid_node.pickle'),
    centroid_connector_link_file = os.path.join(cc_dir, 'cc_link.pickle'),
    centroid_connector_shape_file = os.path.join(cc_dir, 'cc_shape.pickle'),
)

Adding centroid and centroid connector to standard network
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
c:\Users\USYS671257\.conda\envs\wrangler\lib\pickle.py:1718: UserWarning: Unpickling a shapely <2.0 geometry object. Please save the pickle again; shapely 2.1 will not have this compatibility.
  setstate(state)
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:2025: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt

Finished adding centroid and centroid connectors


In [15]:
centroids_df = r_net.links_df[r_net.links_df["centroidconnect"] == True].copy()

centroids_df["bike_access"].value_counts()

bike_access
True     24044
False    17830
Name: count, dtype: int64

### Add Rail access and egress links

In [16]:
r_net = metcouncil_roadway.add_rail_ae_connections(
    r_net,
    metcouncil_parameters
)

cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Creating rail access and egress connection links


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


In [17]:
m_net = metcouncil_roadway.roadway_standard_to_met_council_network(
    r_net,
    metcouncil_parameters    
)

Renaming roadway attributes to be consistent with what metcouncil's model is expecting
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Creating managed lane network.
Separating managed lane links from general purpose links


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\backends\pandas\container.py:544: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  check_obj[col_name] = check_obj[col_name].fillna(
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\array.py:1406: UserWarning: CRS not set for some of the concatenation inputs. Setting output's CRS as WGS 84 (the single non-null crs provided).
  warnings.warn(
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Use

Distance Variable 'distance' already in network. Returning without overwriting.
Finished creating ML lanes variable: ML_lanes
Finished creating hov corridor variable: segment_id
Managed Variable 'managed' already in network. Returning without overwriting.
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Area Type Variable 'area_type' already in network. But some records are missing. Calcualting the missing values without overwriting existing.
Calculating Area Type from Spatial Data and adding as roadway network variable: area_type


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_roadway.py:275: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids_gdf["geometry"] = centroids_gdf["geometry"].centroid


Finished Calculating Area Type from Spatial Data into variable: area_type
Overwriting existing County Variable 'county' already in network
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Adding roadway network variable for county using a spatial join with: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler\metcouncil_data\county\cb_2017_us_county_5m.shp


...

c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\backends\pandas\container.py:544: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  check_obj[col_name] = check_obj[col_name].fillna(
\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\utils\time.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real_end.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\backends\pandas\container.py:544: FutureW

Timespan is not in increasing order: ['19:00', '3:00'].            End time will be treated as next day.
Specified variable to split: ML_lanes not in network variables: Index(['projects', 'rail_only', 'distance', 'source_geometry', 'bike_access',
       'MNPASS_CODE', 'source_model_link_id', 'bus_only', 'B', 'name', 'ref',
       'egress', 'ML_access_point', 'geometry', 'walk_access', 'A', 'managed',
       'access', 'sc_lanes', 'drive_access', 'roadway', 'lanes', 'GP_B',
       'GP_A', 'model_link_id', 'shape_id', 'price', 'ML_projects',
       'ML_egress_point', 'osm_link_id', 'shstReferenceId', 'shstGeometryId',
       'fromIntersectionId', 'toIntersectionId', 'u', 'v', 'nodeIds', 'wayId',
       'roadClass', 'oneWay', 'roundabout', 'link', 'oneway', 'lanes_osm',
       'highway', 'service', 'width', 'maxspeed', 'junction', 'bridge',
       'tunnel', 'landuse', 'area', 'key', 'forward', 'backReferenceId',
       'metadata', 'source', 'county', 'length', 'locationReferences',
       

\\dcclda00dat02.corp.pbwan.net\sag\projects\Met_Council\31000743A\Task 1 Consolidate Architecture\software\network_wrangler\network_wrangler\utils\time.py:233: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  real_end.loc[orig_df["end_time"] < orig_df["start_time"]] += pd.Timedelta(days=1)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\backends\pandas\container.py:544: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  check_obj[col_name] = check_obj[col_name].fillna(
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pandera\backends\pandas\container.py:544: FutureW

Converting variable type to MetCouncil standard
Converting variable type to MetCouncil standard
Setting Coordinate Reference System to EPSG 26915


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


In [18]:
# check if missing IDs
# centroids does not have osm and shst IDs
# centroid connectors does not have osm and shst IDs

# if node missing shst id
print(m_net.nodes_df.shst_node_id.isnull().sum())
print(m_net.nodes_df.shst_node_id.nunique())

# if node missing model node id
print(m_net.nodes_df.model_node_id.nunique())

# if link missing 
print(m_net.links_df.shstReferenceId.isnull().sum())
print(m_net.links_df.shstReferenceId.nunique())
print(m_net.links_df.model_link_id.nunique())

# if link missing node id
print(m_net.links_df.fromIntersectionId.isnull().sum())
print(m_net.links_df.toIntersectionId.isnull().sum())

0
414221
417533
42891
1061504
1104395
21985
21985


In [19]:
m_net.nodes_df.columns

Index(['osm_node_id', 'shst_node_id', 'drive_access', 'walk_access',
       'bike_access', 'model_node_id', 'rail_only', 'X', 'Y', 'projects',
       'geometry', 'GP_model_node_id', 'county', 'N'],
      dtype='object')

In [20]:
m_net.links_df.columns

Index(['projects', 'rail_only', 'distance', 'source_geometry', 'bike_access',
       'MNPASS_CODE', 'source_model_link_id', 'bus_only', 'B', 'name',
       ...
       'price_sov_NT', 'price_hov2_NT', 'price_hov3_NT', 'price_truck_NT',
       'access_EA', 'access_AM', 'access_MD', 'access_PM', 'access_NT',
       'geometry'],
      dtype='object', length=125)

In [21]:
#check column datatype .dtypes
m_net.links_df.MNPASS_CODE.dtype

dtype('int32')

## Populate MnPASS PAY Links


In [22]:
# step 0: make a copy of the metcouncil links dataframe
input_df=(m_net.links_df).copy()

In [23]:
# step 1: select the managed lanes 
code_df = input_df[input_df["MNPASS_CODE"].isin(range(1,100))].copy().reset_index(drop = True)


In [24]:
# step 2: select the connectors
con_df = input_df[input_df["B"].isin(code_df["A"].unique())].copy().reset_index(drop=True)
con_df = con_df[con_df["managed"] == 0].copy().reset_index(drop=True)


In [25]:
# step 3: create a dataframe with the connectors and the MNPASS_PAY value
join_code_df = code_df[["A", "MNPASS_CODE"]].copy().reset_index(drop=True).rename(columns = {"A": "B", "MNPASS_CODE": "MNPASS_PAY_UPDATE"})
join_code_df = join_code_df.merge(con_df[['A','B']].drop_duplicates(), on='B', how='left')

update_con_df = pd.merge(con_df, join_code_df, how="left", on=["A", "B"])
update_con_df = update_con_df[["A", "B", "MNPASS_PAY_UPDATE"]].copy().reset_index(drop=True)


In [26]:
# step 4: merge the MNPASS_PAY_UPDATE values into the original dataframe
output_df = pd.merge(input_df, update_con_df, how="left", on=["A", "B"])
output_df["MNPASS_PAY"] = np.where(output_df["MNPASS_PAY_UPDATE"].isnull(), output_df["MNPASS_PAY"], output_df["MNPASS_PAY_UPDATE"])
output_df = output_df.drop(columns=["MNPASS_PAY_UPDATE"])
m_net.links_df = output_df.copy()

# Write model network as shapefile

In [28]:
#out_cols = ['model_link_id', 'id', 'assign_group', 'drive_access', 'roadway_class',
#            'lanes_AM', 'lanes_MD', 'lanes_PM', 'lanes_NT', 'segment_id', 'HOV', 
#            'price_sov_AM', 'geometry', 'managed']

roadway.write_roadway_as_shp(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    output_link_shp = os.path.join(output_dir2, 'fullnet_v02', 'shapefile', 'BaseLinks.shp'),
    output_node_shp = os.path.join(output_dir2, 'fullnet_v02', 'shapefile', 'BaseNodes.shp'),
    #link_output_variables = out_cols,
    data_to_csv = False,
    data_to_dbf = True,
    export_drive_only = False, # if user only wants drive links/nodes in the shapefile
)

Writing Network as Shapefile
Renaming DBF Node Variables
Renaming variables so that they are DBF-safe


c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\geopandas\geoseries.py:645: FutureWarning: the convert_dtype parameter is deprecated and will be removed in a future version.  Do ``ser.astype(object).apply()`` instead if you want ``convert_dtype=False``.
  result = super().apply(func, convert_dtype=convert_dtype, args=args, **kwargs)


Renaming DBF Link Variables
Renaming variables so that they are DBF-safe
Writing Node Shapes:
 - Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output\fullnet_v02\shapefile\BaseNodes.shp
Writing Link Shapes:
 - Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\consolidate_architecture\output\fullnet_v02\shapefile\BaseLinks.shp


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\cube_wrangler\cube_wrangler\roadway.py:834: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  links_dbf_df.to_file(output_link_shp)
c:\Users\USYS671257\.conda\envs\wrangler\lib\site-packages\pyogrio\raw.py:723: RuntimeWarning: Normalized/laundered field name: 'MNPASS_CODE' to 'MNPASS_COD'
  ogr_write(


# Write model network for Cube

In [29]:
roadway.write_roadway_as_fixedwidth(
    roadway_net = m_net,
    parameters = metcouncil_parameters,
    zones = metcouncil_parameters.zones,
    output_link_txt = os.path.join(output_dir2, 'fullnet_v02', 'links.txt'),
    output_node_txt = os.path.join(output_dir2,  'fullnet_v02','nodes.txt'),
    output_link_header_width_txt = os.path.join(output_dir2,  'fullnet_v02', 'links_header_width.txt'),
    output_node_header_width_txt = os.path.join(output_dir2,  'fullnet_v02','nodes_header_width.txt'),
    output_cube_network_script = os.path.join(output_dir2,  'fullnet_v02',  'make_complete_network_from_fixed_width_file.s'),
)

Starting fixed width conversion
Writing out link database
Writing out link header and width ----
Starting fixed width conversion
Writing out node database
Writing out node header and width


In [30]:
version_02_scenario.transit_net.road_net = version_02_scenario.road_net
standard_transit_net = StandardTransit.fromTransitNetwork(version_02_scenario.transit_net, parameters=metcouncil_parameters)

In [31]:
standard_transit_net = metcouncil_transit.transit_standard_to_met_council_transit_network(
    transit_net = standard_transit_net,
    parameters = metcouncil_parameters,
    line_name_xwalk = os.path.join(output_dir2, 'fullnet_v02', 'line_name_xwalk.csv')
) 

cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler
Converting GTFS Standard Properties to MetCouncil's Cube Standard
cube_wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\cube_wrangler
MetCouncil Wrangler base directory set as: Z:\Met_Council\31000743A\Task 1 Consolidate Architecture\software\met_council_wrangler


\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1253: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])
\\dcclda00dat02.corp.pbwan.net\sag\projects\met_council\31000743a\task 1 consolidate architecture\software\met_council_wrangler\met_council_wrangler\metcouncil_transit.py:1265: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  trip_df.groupby(["agency_id", "route_id", "direction_id", "shp_index"])


In [32]:
standard_transit_net.write_as_cube_lin(outpath = os.path.join(output_dir2, 'fullnet_v02', 'transit.lin'))